# Notebook for data exploration

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from nilearn import datasets

sns.set_theme(style="whitegrid")

## Load one preprocessed ABIDE subject

Load the phenotypic data and CC200 regional time series.

In [ ]:
# Set N_SUBJECTS = None to download all available subjects.
N_SUBJECTS = 1
ABIDE_DATA_DIR = Path("data") / "abide_pcp"

abide = datasets.fetch_abide_pcp(
    data_dir=ABIDE_DATA_DIR,
    n_subjects=N_SUBJECTS,
    pipeline="cpac",
    band_pass_filtering=True,
    global_signal_regression=False,
    derivatives=["rois_cc200"],
    quality_checked=True,
)

phenotypic = abide.phenotypic
cc200_time_series = abide.rois_cc200

[fetch_abide_pcp] Dataset directory found: data\abide_pcp\ABIDE_pcp
Loaded 1 subjects


,i,Unnamed: 0,SUB_ID,X,subject,SITE_ID,FILE_ID,DX_GROUP,DSM_IV_TR,AGE_AT_SCAN,...,qc_notes_rater_1,qc_anat_rater_2,qc_anat_notes_rater_2,qc_func_rater_2,qc_func_notes_rater_2,qc_anat_rater_3,qc_anat_notes_rater_3,qc_func_rater_3,qc_func_notes_rater_3,SUB_IN_SMP
1,1,2,50003,2,50003,PITT,Pitt_0050003,1,1,24.45,...,NaN,OK,NaN,OK,NaN,OK,NaN,OK,NaN,1


## Phenotypic Data
<div style="font-size: 15px; line-height: 1.35;">
Missing values: `NaN` / `-9999`

**IDs + basic information (13 features)**
- `i`, `Unnamed: 0`, `X`: old CSV indexes; ignore
- `SUB_ID`, `subject`, `FILE_ID`: participant/file IDs
- `SITE_ID`: scanning site, e.g. `PITT`
- `DX_GROUP`: `1 = autism`, `2 = control`
- `DSM_IV_TR`: `0 = control`, `1 = autism`, `2 = Asperger`, `3 = PDD-NOS`, `4 = Asperger/PDD-NOS`
- `AGE_AT_SCAN`: age in years
- `SEX`: `1 = male`, `2 = female`
- Handedness (2): `R`, `L`, `Mixed`, `Ambi`, `L->R`; score from `-100` to `100`

**IQ (6 features)**  
`FIQ`, `VIQ`, `PIQ`: full-scale, verbal, nonverbal IQ + test types (`WASI`, `WISC`, `WAIS`, ...).

**Autism assessments (24 features)**
- `ADI_R_*` (5): social, communication, repetitive behaviour, onset, reliability
- `ADOS_*` (10): observed behaviour, total, module, severity
- `SRS_*` (7): social awareness, cognition, communication, motivation, mannerisms
- `SCQ_TOTAL`, `AQ_TOTAL`: questionnaire totals

**Medication + other diagnoses (4 features)**  
`COMORBIDITY`, medication status/name, stimulants stopped before scan (`0 = no`, `1 = yes`).

**`VINELAND_*` (15 features)**  
Communication, daily living, social skills, coping, adaptive behaviour.

**`WISC_IV_*` (14 features)**  
Reasoning, working memory, processing speed + cognitive subtests.

**Scan information (3 features)**  
Eyes (`1 = open`, `2 = closed`), age at anatomical scan, BMI.

**Image quality (16 features)**
- `anat_*` (6): contrast, noise, smoothness, artifacts
- `func_*` (10): signal quality, outliers, head motion

**Manual quality control (10 features)**  
`qc_*`: human ratings/notes; `OK`, `maybe`, `fail`.

**Original sample (1 feature)**  
`SUB_IN_SMP`: included in original sample; `0 = no`, `1 = yes`.
</div>

In [ ]:
print("\nPhenotypic data of one subject:")
phenotypic.head()


Phenotypic data:


,i,Unnamed: 0,SUB_ID,X,subject,SITE_ID,FILE_ID,DX_GROUP,DSM_IV_TR,AGE_AT_SCAN,...,qc_notes_rater_1,qc_anat_rater_2,qc_anat_notes_rater_2,qc_func_rater_2,qc_func_notes_rater_2,qc_anat_rater_3,qc_anat_notes_rater_3,qc_func_rater_3,qc_func_notes_rater_3,SUB_IN_SMP
1,1,2,50003,2,50003,PITT,Pitt_0050003,1,1,24.45,...,NaN,OK,NaN,OK,NaN,OK,NaN,OK,NaN,1


## Inspect the CC200 regional time series

Rows are scan time points and columns are the 200 CC200 brain regions.

In [4]:
time_series_source = cc200_time_series[0]

time_series = np.asarray(time_series_source)

print(f"Time-series shape: {time_series.shape}")
print(f"Time points: {time_series.shape[0]}")
print(f"Brain regions: {time_series.shape[1]}")

time_series

Time-series shape: (196, 200)
Time points: 196
Brain regions: 200


array([[  24.560342,  -18.407722,   38.447947, ...,    7.148534,
         -16.70126 ,   -9.040049],
       [  12.432386,  -24.225554,   32.72212 , ...,    6.915863,
         -18.816158,  -16.079462],
       [ -15.628296,  -26.657624,    4.821955, ...,    1.262588,
         -17.565622,  -29.46237 ],
       ...,
       [ -78.053982,  -28.795672, -114.368604, ...,  -34.754758,
         -96.55319 ,  -78.052758],
       [ -46.00144 ,  -16.829028,  -64.951275, ...,  -17.228399,
         -49.759665,  -57.640293],
       [ -11.677131,   -9.211697,  -21.498266, ...,    0.149923,
          -8.384009,  -29.650198]], shape=(196, 200))